# 01 · Build the training set

ExoSeer — Bharatiya Antariksh Hackathon 2026.

Download the **NASA Exoplanet Archive Kepler KOI cumulative table**, build a feature
matrix whose columns match `exoseer_core.vet.MODEL_FEATURES`, and label rows from
`koi_disposition` (+ `koi_fpflag_ss` to separate eclipsing binaries). Output is vendored
to `backend/data/koi_cumulative.csv` and a ready-to-train `(X, y)`.

> The live pipeline computes the equivalent features from TLS + a stellar lookup, so the
> training and inference feature vectors are aligned by construction.


In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../backend')
from app.exoseer_core.vet import MODEL_FEATURES, CLASSES
print('features:', MODEL_FEATURES)
print('classes :', CLASSES)


### Download the KOI cumulative table (TAP service)


In [ ]:
TAP = ('https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query='
       'select+koi_disposition,koi_period,koi_duration,koi_depth,koi_model_snr,'
       'koi_prad,koi_srad,koi_steff,koi_slogg,koi_fpflag_ss,koi_fpflag_nt,'
       'koi_fpflag_co,koi_fpflag_ec+from+cumulative&format=csv')
df = pd.read_csv(TAP)
df.to_csv('../backend/data/koi_cumulative.csv', index=False)
print(df.shape); df['koi_disposition'].value_counts()


### Label rows
- `CONFIRMED` → **planet_candidate**
- `FALSE POSITIVE` with the stellar-eclipse flag → **eclipsing_binary**; else **variable_or_other**
- `CANDIDATE` rows are dropped (ambiguous).


In [ ]:
def label_row(disp, ss):
    if disp == 'CONFIRMED': return 'planet_candidate'
    if disp == 'FALSE POSITIVE': return 'eclipsing_binary' if int(ss or 0)==1 else 'variable_or_other'
    return None
df['label'] = [label_row(d, s) for d, s in zip(df.koi_disposition, df.koi_fpflag_ss)]
df = df[df.label.notna()]
df.label.value_counts()


### Build the feature matrix (must match `MODEL_FEATURES`)


In [ ]:
feat = pd.DataFrame({
    'period': df.koi_period, 'duration_hours': df.koi_duration, 'depth_ppm': df.koi_depth,
    'snr': df.koi_model_snr, 'implied_radius_rearth': df.koi_prad,
    'srad': df.koi_srad, 'steff': df.koi_steff, 'slogg': df.koi_slogg })
feat['duration_over_period'] = (feat.duration_hours/24)/feat.period
feat['depth_snr'] = feat.depth_ppm * feat.snr
feat = feat[MODEL_FEATURES].replace([np.inf,-np.inf], np.nan)
feat = feat.fillna(feat.median(numeric_only=True))
X, y = feat.to_numpy(float), df.label.to_numpy()
np.save('X.npy', X); np.save('y.npy', y)
print('X', X.shape, '| y', np.unique(y, return_counts=True))


Proceed to **02_train_vetting_model.ipynb**.
